In [51]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

In [52]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

In [53]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

### 代码功能解释

#### 第一行代码：`os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`
1. **具体修改**：
    设置系统环境变量 `HF_ENDPOINT`，将 Hugging Face 的官方下载源地址强制替换为国内镜像源地址 `https://hf-mirror.com`。
2. **为什么要用镜像源而非官方源**：
    - **网络访问限制**：国内网络直接访问 Hugging Face 官方源（`huggingface.co`）通常速度极慢，甚至存在连接超时、下载失败的问题。
    - **加速下载**：使用国内镜像源（如清华镜像 `hf-mirror.com`）可以大幅提升数据集和模型文件的下载速度，保证 `load_dataset` 能顺利执行。
    - **稳定性**：镜像源同步了官方的所有数据，兼容性与官方源一致，是解决海外源访问困难的首选方案。

#### 第二行代码：`dataset = datasets.load_dataset("bentrevett/multi30k")`
1. **具体操作**：
    使用 Hugging Face Datasets 库的 `load_dataset` 函数，直接从远程仓库/服务器加载名为 `bentrevett/multi30k` 的机器翻译数据集。
2. **完成功能**：
    - 自动下载并缓存 `multi30k` 数据集（包含德→英的翻译语料）。
    - 返回一个包含 **训练集 (`train`)、验证集 (`validation`)、测试集 (`test`)** 的 `DatasetDict` 对象。
    - 数据集加载完成后，即可后续对数据进行分词、构建词汇表、编码等预处理操作。

## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [54]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [55]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [56]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [57]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [58]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


In [59]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

### tokenize_example 函数说明
`tokenize_example` 是德英翻译任务的数据预处理函数，用于对平行语料进行分词和标准化处理。

函数接收单条语料样本、分词模型、最大长度、大小写开关和序列标记，先使用 spaCy 模型对英语和德语文本分词并截断超长文本，再根据参数将所有词元转为小写，最后在序列首尾添加起始标记和结束标记，返回处理好的英德分词序列，为 Seq2Seq 模型训练提供规范数据。

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


In [60]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

###  `<sos>` 与 `<eos>` 的含义
- **`<sos>`**：全称 *Start of Sequence*，是**序列起始标记**，表示句子的开始。
- **`<eos>`**：全称 *End of Sequence*，是**序列结束标记**，表示句子的结束。
- 作用：让 Seq2Seq 模型明确感知句子的边界，是机器翻译中解码器生成句子的重要信号。

---

###  `map` 函数的作用
- `map` 是 Hugging Face Datasets 的数据集处理方法，用于**对数据集中的每一条样本应用同一个预处理函数**（这里是 `tokenize_example`）。
- 它会自动遍历训练集、验证集、测试集，批量完成分词、加 `<sos>/<eos>` 等统一预处理，提高效率，避免手动循环。

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [94]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [62]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [63]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [64]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [65]:
en_vocab["the"]

7

In [66]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [67]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [68]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [69]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

### <unk> 是 Unknown Token（未知词元） 的缩写。当某个词（如 crime）不在训练集构建的词汇表（Vocab）中时，模型无法识别它，就会用统一的 <unk> 标记来替代，保证序列能正常输入模型训练。

### `<unk>`是 Unknown Token（未知词元） 的缩写。当某个词（如 crime）不在训练集构建的词汇表（Vocab）中时，模型无法识别它，就会用统一的 `<unk>` 标记来替代，保证序列能正常输入模型训练。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


In [70]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [71]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### numericalize_example 函数
- 这是机器翻译任务的数值化预处理函数，作用是把分词后的 token 文本序列，转换成神经网络模型可以直接计算的数字索引序列。
- 调用词汇表的 lookup_indices 方法，将英语、德语的 token 列表分别映射为对应的索引 ID。
- 以字典形式返回处理后的英 / 德索引序列，为后续模型训练做数据准备。
### map 批量处理
##### **map** 是 Hugging Face Datasets 库的数据集处理方法，用于对数据集的每一条样本批量应用预处理函数。
- 这里将 numericalize_example 函数，批量应用到训练集、验证集、测试集的所有样本上。
- 高效完成全数据集的数值化转换，统一数据格式，让数据符合 Seq2Seq 模型的输入要求。
- 下方的进度条显示了批量处理的完成状态，验证了数据预处理的成功执行。

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [72]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [73]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [74]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

In [75]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [76]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [77]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

In [78]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

## Encoder 类说明
Encoder 是 Seq2Seq 模型的编码器，负责把德语句子编码成语义向量，送给解码器生成英语句子。

### 初始化方法 __init__
- input_dim：德语词汇表大小
- embedding_dim：词向量维度
- hidden_dim：LSTM 隐藏层维度
- n_layers：LSTM 层数
- dropout：防止过拟合的随机失活率

内部构建了三层结构：词嵌入层将索引转为向量，LSTM 层处理序列信息，Dropout 层随机失活神经元。

### 前向传播 forward
输入是德语句子的索引序列。
先经过嵌入层和 Dropout 处理，再输入 LSTM 进行编码。
最终返回 LSTM 最后的隐藏状态和细胞状态，这两个向量包含整句德语的语义信息，作为解码器的初始输入。

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

In [79]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

## Decoder 类说明
Decoder 是 Seq2Seq 模型的解码器，负责根据编码器提供的语义向量，逐词生成英语翻译句子。

#### 初始化方法 __init__
- input_dim：英语词汇表大小
- embedding_dim：词向量维度
- hidden_dim：LSTM 隐藏层维度
- output_dim：最终输出的词汇表大小
- n_layers：LSTM 层数
- dropout：随机失活率，防止过拟合

内部包含词嵌入层、LSTM 层、全连接输出层和 Dropout 层。

#### 前向传播 forward
输入为上一步生成的单词、编码器传来的隐藏状态和细胞状态。
先对输入进行维度扩展并通过嵌入层与 Dropout 处理。
将结果输入 LSTM，结合上下文信息更新隐藏状态与细胞状态。
最后通过全连接层预测下一个单词的概率分布。

输出当前单词的预测结果、新的隐藏状态和细胞状态，用于下一步继续生成单词，直到句子完成。

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

In [80]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

## Seq2Seq 类代码说明

---

### 一、整体作用
`Seq2Seq` 是**机器翻译任务的端到端模型封装类**，将编码器（Encoder）和解码器（Decoder）整合为完整的翻译 pipeline，实现从源语言（德语）到目标语言（英语）的端到端翻译，并集成 **Teacher Forcing** 机制优化训练。

---

### 二、`__init__` 初始化方法
- 接收编码器、解码器和运行设备作为输入，完成模型组件的绑定。
- 增加断言校验：强制编码器与解码器的**隐藏层维度**、**堆叠层数**完全一致，保证编码-解码流程的维度匹配，避免运行错误。
- 保存编码器、解码器和设备信息，为前向传播做准备。

---

### 三、`forward` 前向传播流程
1.  **输入接收**
    - `src`：源语言（德语）输入序列
    - `trg`：目标语言（英语）真实标签序列（用于训练）
    - `teacher_forcing_ratio`：Teacher Forcing 概率，控制训练时解码器的输入来源

2.  **编码器编码**
    将源语言序列输入编码器，得到最终的隐藏状态 `hidden` 和细胞状态 `cell`，作为源语言的语义向量，初始化解码器。

3.  **解码器逐词生成（结合 Teacher Forcing）**
    - 初始输入为目标语言的 `<sos>` 起始标记
    - 每一步根据 `teacher_forcing_ratio` 随机决定输入：
      - 以 `teacher_forcing_ratio` 的概率，使用**真实标签（Teacher Forcing）**作为下一步输入，加速模型收敛
      - 以 `1 - teacher_forcing_ratio` 的概率，使用**上一步预测结果**作为下一步输入，模拟推理场景
    - 解码器逐词生成预测，将结果存入输出张量

4.  **输出返回**
    返回解码器的完整预测序列，用于计算损失、更新模型参数。

---

### 四、Teacher Forcing 机制说明
Teacher Forcing 是 Seq2Seq 模型的经典训练优化技巧：
- 训练时，以一定概率直接使用**真实标签**作为解码器的下一步输入，而非上一步的预测结果
- 解决了传统自回归训练中“误差累积”的问题，大幅加快模型收敛速度，提升训练稳定性
- 推理时关闭该机制，完全使用上一步预测结果生成句子，保证翻译流程的一致性

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [97]:
# 超参数配置
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [82]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [83]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [84]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [85]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [98]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    # 🔹 将模型切换为训练模式（启用 Dropout、BatchNorm 等训练专属层）
    model.train()
    # 🔹 初始化本轮 epoch 的总损失为 0
    epoch_loss = 0
    # 🔹 遍历数据加载器中的每一个 batch
    for i, batch in enumerate(data_loader):
        # 🔹 将源语言（德语）的索引序列放到指定设备（GPU/CPU）
        src = batch["de_ids"].to(device)
        # 🔹 将目标语言（英语）的索引序列放到指定设备
        trg = batch["en_ids"].to(device)

        # 🔹 梯度清零，避免上一轮梯度累积影响本轮训练
        optimizer.zero_grad()
        # 🔹 前向传播：输入源序列、目标序列，启用 Teacher Forcing，得到模型预测输出
        output = model(src, trg, teacher_forcing_ratio)

        # 🔹 提取输出的最后一维（目标语言词汇表大小）
        output_dim = output.shape[-1]
        # 🔹 调整输出形状：去掉 <sos> 起始 token，展平为二维张量，适配损失函数输入
        output = output[1:].view(-1, output_dim)
        # 🔹 调整目标序列形状：去掉 <sos>，展平为一维张量，与输出对齐
        trg = trg[1:].view(-1)

        # 🔹 计算预测值与真实值的损失
        loss = criterion(output, trg)
        # 🔹 反向传播，计算梯度
        loss.backward()
        # 🔹 梯度裁剪，防止梯度爆炸，clip 为最大梯度范数
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        # 🔹 更新模型参数
        optimizer.step()

        # 🔹 累加本轮 batch 的损失到 epoch 总损失
        epoch_loss += loss.item()
    # 🔹 返回本轮 epoch 的平均损失（总损失 / batch 数量）
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [99]:
def evaluate_fn(model, data_loader, criterion, device):
    # 🔹 将模型切换为评估模式（禁用 Dropout、BatchNorm 等，保证推理稳定）
    model.eval()
    # 🔹 初始化本轮 epoch 的总损失为 0
    epoch_loss = 0
    # 🔹 禁用梯度计算，减少内存占用、加速推理（评估无需更新参数）
    with torch.no_grad():
        # 🔹 遍历数据加载器中的每一个 batch
        for i, batch in enumerate(data_loader):
            # 🔹 将源语言（德语）的索引序列放到指定设备
            src = batch["de_ids"].to(device)
            # 🔹 将目标语言（英语）的索引序列放到指定设备
            trg = batch["en_ids"].to(device)

            # 🔹 前向传播：关闭 Teacher Forcing（teacher_forcing_ratio=0），完全自回归生成
            output = model(src, trg, 0)

            # 🔹 提取输出的最后一维（目标语言词汇表大小）
            output_dim = output.shape[-1]
            # 🔹 调整输出形状：去掉 <sos>，展平为二维张量
            output = output[1:].view(-1, output_dim)
            # 🔹 调整目标序列形状：去掉 <sos>，展平为一维张量
            trg = trg[1:].view(-1)

            # 🔹 计算预测值与真实值的损失
            loss = criterion(output, trg)
            # 🔹 累加本轮 batch 的损失到 epoch 总损失
            epoch_loss += loss.item()
    # 🔹 返回本轮 epoch 的平均损失
    return epoch_loss / len(data_loader)

# 模型训练

In [88]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [14:17<00:00, 857.41s/it]

	Train Loss:   5.046 | Train PPL: 155.447
	Valid Loss:   4.989 | Valid PPL: 146.771


# 模型验证

In [89]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [90]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [91]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [92]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [93]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'a',
 'shirt',
 'is',
 'a',
 'a',
 '.',
 '.',
 '<eos>']